In [ ]:
import random

class Environment:
    def __init__(self, state='Dirty'):
        self.state = state

    def get_percept(self):
        return self.state

    def clean_room(self):
        self.state = 'Clean'
        return 10

    def no_action_reward(self):
        return 0

class LearningBasedAgent:
    def __init__(self, actions):
        self.Q = {}
        self.actions = actions
        self.alpha = 0.1     # Learning rate
        self.gamma = 0.9     # Discount factor
        self.epsilon = 0.1   # Exploration rate (10% chance to act randomly)

    def get_Q_value(self, state, action):
        # Returns the Q-value if it exists, otherwise defaults to 0.0
        return self.Q.get((state, action), 0.0)

    def select_action(self, state):
        # Epsilon-Greedy approach: Explore vs Exploit   
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions) # Explore: Pick a random action
        else:
            # Exploit: Pick the action with the highest Q-value for this state
            return max(self.actions, key=lambda a: self.get_Q_value(state, a))
 
    def learn(self, state, action, reward, next_state):
        # The Q-Learning formula
        old_Q = self.get_Q_value(state, action)
        best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
        
        # Calculate the new score and save it in the Q-tabl
        new_Q = old_Q + self.alpha * (reward + self.gamma * best_future_Q - old_Q)
        self.Q[(state, action)] = new_Q

    def act(self, state):
        return self.select_action(state)

def run_agent(agent, environment, steps):
    for step in range(steps):
        percept = environment.get_percept()
        action = agent.act(percept)
        
        # --- THE FIX ---
        # The environment only cleans and rewards IF the agent actually chose to clean
        if action == 'Clean the room' and percept == 'Dirty':
            reward = environment.clean_room()
        else:
            # If the agent does nothing, or tries to clean an already clean room
            reward = environment.no_action_reward()
        # ---------------
            
        print(f"Step {step + 1}: Percept - '{percept}', Action - '{action}', Reward - {reward}")
        
        next_percept = environment.get_percept()
        
        # The agent updates its brain based on what just happened
        agent.learn(percept, action, reward, next_percept)


# --- Execution ---

# Define the possible actions
actions = ['Clean the room', 'No action needed']

# Create instances of agent and environment
agent = LearningBasedAgent(actions)
environment = Environment()

# Run the agent in the environment for 10 steps 
# (Increased from 5 to give it a tiny bit more time to learn)
print("--- Starting Simulation ---")
run_agent(agent, environment, 10)

# Print the agent's memory (Q-table) to see what it learned
print("\n--- Agent's Q-Table (Memory) ---")
if not agent.Q:
    print("Q-table is empty.")
else:
    for state_action_pair, score in agent.Q.items():
        state = state_action_pair[0]
        action = state_action_pair[1]
        print(f"When '{state}', doing '{action}' has a score of: {score:.4f}")

--- Starting Simulation ---
Steps: 1 -> Action: Clean the room -> Percept: Dirty -> Reward: 10
Steps: 2 -> Action: No action needed -> Percept: Clean -> Reward: 0
Steps: 3 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 4 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 5 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 6 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 7 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 8 -> Action: No action needed -> Percept: Clean -> Reward: 0
Steps: 9 -> Action: Clean the room -> Percept: Clean -> Reward: 0
Steps: 10 -> Action: Clean the room -> Percept: Clean -> Reward: 0

--- Agent's Q-Table (Memory) ---
When Dirty doing Clean the room has a score of 1.0000
When Clean doing No action needed has a score of 0.0000
When Clean doing Clean the room has a score of 0.0000
